# Motion Pickup Debug Demo

Hardcoded floor-demo flow without camera or DepthNet startup. The base uses fixed scan, approach, and drop-drive timing. Arm states are loaded from arm_grab_tuning_params.json.

In this integrated demo, ready_s* is treated as reaching_state: it lowers the arm while leaving servo 4 alone. The gripper is opened before reaching_state. lift_state_grip only moves servo 4, and lift_state_carry only raises servos 1/2/3/5.

In [11]:
import json
import time
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

GRAB_PARAM_PATH = Path('arm_grab_tuning_params.json')
MOTION_PARAM_PATH = Path('motion_pickup_demo_params.json')

DEFAULT_GRAB_PARAMS = {
    'safe_s1': -11,
    'safe_s2': -77,
    'safe_s3': -65,
    'safe_s4': -14,
    'safe_s5': -26,
    'ready_s1': -20,
    'ready_s2': 96,
    'ready_s3': 34,
    'ready_s4': 0,
    'ready_s5': 0,
    'lift_state_s1': -21,
    'lift_state_s2': -4,
    'lift_state_s3': -6,
    'lift_state_s4': -42,
    'lift_state_s5': 0,
    'gripper_open': 0,
    'arm_speed': 80,
    'gripper_speed': 120,
    'settle_seconds': 0.35,
    'state_pause_seconds': 0.50,
    'reaching_pause_seconds': 2.00,
}

DEFAULT_MOTION_PARAMS = {
    'dry_run_base': True,
    'dry_run_arm': True,
    'start_x': 0.0,
    'start_y': 0.0,
    'can_x': 1.0,
    'can_y': 0.0,
    'drop_x': 1.0,
    'drop_y': 1.0,
    'scan_turn_direction': 'left',
    'scan_turn_speed': 0.12,
    'scan_turn_seconds': 1.0,
    'scan_stop_angle_deg': 90,
    'approach_direction': 'forward',
    'approach_speed': 0.15,
    'approach_seconds': 1.2,
    'approach_distance_m': 0.5,
    'drop_turn_direction': 'right',
    'drop_turn_speed': 0.12,
    'drop_turn_seconds': 1.0,
    'drop_stop_angle_deg': 90,
    'drop_drive_direction': 'forward',
    'drop_drive_speed': 0.15,
    'drop_drive_seconds': 1.2,
    'drop_drive_distance_m': 0.5,
}


def load_json_params(path, defaults):
    params = dict(defaults)
    if path.exists():
        params.update(json.loads(path.read_text()))
    return params


grab_params = load_json_params(GRAB_PARAM_PATH, DEFAULT_GRAB_PARAMS)
motion_params = load_json_params(MOTION_PARAM_PATH, DEFAULT_MOTION_PARAMS)

print('[params] grab:', GRAB_PARAM_PATH if GRAB_PARAM_PATH.exists() else 'defaults')
print('[params] motion:', MOTION_PARAM_PATH if MOTION_PARAM_PATH.exists() else 'defaults')

[params] grab: arm_grab_tuning_params.json
[params] motion: motion_pickup_demo_params.json


In [12]:
def int_slider(name, params, min_value=-180, max_value=180, step=1, width='520px'):
    return widgets.IntSlider(
        value=int(params[name]), min=min_value, max=max_value, step=step,
        description=name, continuous_update=False,
        style={'description_width': '170px'}, layout=widgets.Layout(width=width)
    )


def float_slider(name, params, min_value=0.0, max_value=2.0, step=0.01, width='520px'):
    return widgets.FloatSlider(
        value=float(params[name]), min=min_value, max=max_value, step=step,
        description=name, continuous_update=False, readout_format='.2f',
        style={'description_width': '170px'}, layout=widgets.Layout(width=width)
    )


def direction_dropdown(name, params):
    return widgets.Dropdown(
        value=params[name], options=['forward', 'backward', 'left', 'right'],
        description=name, style={'description_width': '170px'}, layout=widgets.Layout(width='360px')
    )


def turn_dropdown(name, params):
    return widgets.Dropdown(
        value=params[name], options=['left', 'right'],
        description=name, style={'description_width': '170px'}, layout=widgets.Layout(width='360px')
    )


grab_widgets = {
    'state_pause_seconds': float_slider('state_pause_seconds', grab_params, 0.0, 10.0, 0.05),
    'reaching_pause_seconds': float_slider('reaching_pause_seconds', grab_params, 0.0, 10.0, 0.05),
}

motion_widgets = {
    'dry_run_base': widgets.Checkbox(value=bool(motion_params['dry_run_base']), description='dry_run_base'),
    'dry_run_arm': widgets.Checkbox(value=bool(motion_params['dry_run_arm']), description='dry_run_arm'),
    'start_x': float_slider('start_x', motion_params, -5.0, 5.0, 0.05),
    'start_y': float_slider('start_y', motion_params, -5.0, 5.0, 0.05),
    'can_x': float_slider('can_x', motion_params, -5.0, 5.0, 0.05),
    'can_y': float_slider('can_y', motion_params, -5.0, 5.0, 0.05),
    'drop_x': float_slider('drop_x', motion_params, -5.0, 5.0, 0.05),
    'drop_y': float_slider('drop_y', motion_params, -5.0, 5.0, 0.05),
    'scan_turn_direction': turn_dropdown('scan_turn_direction', motion_params),
    'scan_turn_speed': float_slider('scan_turn_speed', motion_params, 0.0, 0.5, 0.01),
    'scan_turn_seconds': float_slider('scan_turn_seconds', motion_params, 0.0, 5.0, 0.05),
    'scan_stop_angle_deg': int_slider('scan_stop_angle_deg', motion_params, 0, 360, 5),
    'approach_direction': direction_dropdown('approach_direction', motion_params),
    'approach_speed': float_slider('approach_speed', motion_params, 0.0, 0.5, 0.01),
    'approach_seconds': float_slider('approach_seconds', motion_params, 0.0, 8.0, 0.05),
    'approach_distance_m': float_slider('approach_distance_m', motion_params, 0.0, 5.0, 0.05),
    'drop_turn_direction': turn_dropdown('drop_turn_direction', motion_params),
    'drop_turn_speed': float_slider('drop_turn_speed', motion_params, 0.0, 0.5, 0.01),
    'drop_turn_seconds': float_slider('drop_turn_seconds', motion_params, 0.0, 5.0, 0.05),
    'drop_stop_angle_deg': int_slider('drop_stop_angle_deg', motion_params, 0, 360, 5),
    'drop_drive_direction': direction_dropdown('drop_drive_direction', motion_params),
    'drop_drive_speed': float_slider('drop_drive_speed', motion_params, 0.0, 0.5, 0.01),
    'drop_drive_seconds': float_slider('drop_drive_seconds', motion_params, 0.0, 8.0, 0.05),
    'drop_drive_distance_m': float_slider('drop_drive_distance_m', motion_params, 0.0, 5.0, 0.05),
}


def current_grab_params():
    data = dict(grab_params)
    data['state_pause_seconds'] = float(grab_widgets['state_pause_seconds'].value)
    data['reaching_pause_seconds'] = float(grab_widgets['reaching_pause_seconds'].value)
    return data


def current_motion_params():
    return {name: widget.value for name, widget in motion_widgets.items()}


def save_grab_params():
    data = current_grab_params()
    grab_params.update(data)
    GRAB_PARAM_PATH.write_text(json.dumps(data, indent=2) + '\n')
    print('[params] saved grab params to', GRAB_PARAM_PATH)


def save_motion_params():
    data = current_motion_params()
    motion_params.update(data)
    MOTION_PARAM_PATH.write_text(json.dumps(data, indent=2) + '\n')
    print('[params] saved motion params to', MOTION_PARAM_PATH)


def save_all_params(_=None):
    save_grab_params()
    save_motion_params()

In [13]:
robot = None
ttl_servo = None
operation_busy = False
control_buttons = []


def ensure_robot():
    global robot
    if motion_widgets['dry_run_base'].value:
        return None
    if robot is None:
        from jetbot import Robot
        robot = Robot()
        print('[base] Robot connected')
    return robot


def ensure_servos():
    global ttl_servo
    if motion_widgets['dry_run_arm'].value:
        return None
    if ttl_servo is None:
        from SCSCtrl import TTLServo
        ttl_servo = TTLServo
        print('[arm] TTLServo connected')
    return ttl_servo


def set_controls_disabled(disabled):
    for button in control_buttons:
        button.disabled = disabled


def base_stop():
    if robot is not None:
        robot.stop()
    print('[base] stop')


def base_drive(direction, speed, seconds, label):
    bot = ensure_robot()
    print('[base] {} direction={} speed={} seconds={}'.format(label, direction, speed, seconds))
    if bot is not None:
        try:
            if direction == 'forward':
                bot.forward(float(speed))
            elif direction == 'backward':
                bot.backward(float(speed))
            elif direction == 'left':
                bot.left(float(speed))
            elif direction == 'right':
                bot.right(float(speed))
            else:
                raise ValueError('unknown base direction: {}'.format(direction))
            time.sleep(float(seconds))
        finally:
            bot.stop()
    else:
        time.sleep(float(seconds))
    print('[base] {} done'.format(label))


def grab_value(name):
    if name in grab_widgets:
        return float(grab_widgets[name].value)
    return grab_params[name]


def arm_speed():
    return grab_value('arm_speed')


def gripper_speed():
    return grab_value('gripper_speed')


def move_servo(servo_id, angle, speed, label=''):
    servos = ensure_servos()
    print('[arm] servo={} angle={} speed={} {}'.format(servo_id, angle, speed, label))
    if servos is not None:
        servos.servoAngleCtrl(int(servo_id), int(angle), 1, int(speed))
    time.sleep(float(grab_value('settle_seconds')))


def apply_pose(name, pose, speed):
    print('[arm] pose:', name)
    for servo_id, angle in pose:
        move_servo(servo_id, angle, speed, name)


def state_pause(label):
    seconds = float(grab_value('state_pause_seconds'))
    if seconds > 0:
        print('[arm] state pause after {}: {}s'.format(label, seconds))
        time.sleep(seconds)


def reaching_pause(label):
    seconds = float(grab_value('reaching_pause_seconds'))
    if seconds > 0:
        print('[arm] reaching pause after {} before gripper action: {}s'.format(label, seconds))
        time.sleep(seconds)


def enter_state(name, detail=''):
    stamp = time.strftime('%H:%M:%S')
    if detail:
        print('[state] {} enter {} | {}'.format(stamp, name, detail))
    else:
        print('[state] {} enter {}'.format(stamp, name))


def format_pose(pose):
    return ', '.join('s{}={}'.format(servo_id, angle) for servo_id, angle in pose)


def safe_home():
    pose = [(1, grab_value('safe_s1')), (2, grab_value('safe_s2')), (3, grab_value('safe_s3')), (4, grab_value('safe_s4')), (5, grab_value('safe_s5'))]
    enter_state('safe_home', format_pose(pose))
    apply_pose('safe_home', pose, arm_speed())


def open_gripper(label='open_gripper'):
    enter_state(label, 's4={}'.format(grab_value('gripper_open')))
    move_servo(4, grab_value('gripper_open'), gripper_speed(), label)


def reaching_state(label='reaching_state'):
    pose = [(1, grab_value('ready_s1')), (2, grab_value('ready_s2')), (3, grab_value('ready_s3')), (5, grab_value('ready_s5'))]
    enter_state(label, '{}; s4 unchanged'.format(format_pose(pose)))
    apply_pose(label, pose, arm_speed())


def lift_state_grip():
    enter_state('lift_state_grip', 's4={} only'.format(grab_value('lift_state_s4')))
    move_servo(4, grab_value('lift_state_s4'), gripper_speed(), 'lift_state_grip')


def lift_state_carry():
    pose = [(5, grab_value('lift_state_s5')), (1, grab_value('lift_state_s1')), (2, grab_value('lift_state_s2')), (3, grab_value('lift_state_s3'))]
    enter_state('lift_state_carry', '{}; s4 unchanged'.format(format_pose(pose)))
    apply_pose('lift_state_carry', pose, arm_speed())


def lift_state():
    lift_state_grip()
    state_pause('lift_state_grip')
    lift_state_carry()


def grab_sequence_without_final_home():
    print('[flow] grab sequence using', GRAB_PARAM_PATH)
    open_gripper()
    state_pause('open_gripper')
    reaching_state()
    reaching_pause('reaching_state')
    lift_state_grip()
    state_pause('lift_state_grip')
    lift_state_carry()
    state_pause('lift_state_carry')


def drop_release_sequence():
    print('[flow] lower to drop using reaching_state angles')
    reaching_state('drop_reaching_state')
    reaching_pause('drop_reaching_state')
    open_gripper('drop_release')
    state_pause('drop_release')

In [14]:
def print_points():
    p = current_motion_params()
    print('[points] A start=({:.2f}, {:.2f}) B can=({:.2f}, {:.2f}) C drop=({:.2f}, {:.2f})'.format(
        p['start_x'], p['start_y'], p['can_x'], p['can_y'], p['drop_x'], p['drop_y']
    ))
    print('[move] scan stop angle={} deg, approach distance={} m'.format(p['scan_stop_angle_deg'], p['approach_distance_m']))
    print('[move] drop turn stop angle={} deg, drop drive distance={} m'.format(p['drop_stop_angle_deg'], p['drop_drive_distance_m']))


def scan_for_can():
    p = current_motion_params()
    enter_state('scan_for_can', 'direction={} speed={} seconds={} stop_angle={}deg'.format(p['scan_turn_direction'], p['scan_turn_speed'], p['scan_turn_seconds'], p['scan_stop_angle_deg']))
    base_drive(p['scan_turn_direction'], p['scan_turn_speed'], p['scan_turn_seconds'], 'scan_turn_to_angle_{}'.format(p['scan_stop_angle_deg']))


def approach_can():
    p = current_motion_params()
    enter_state('approach_can', 'direction={} speed={} seconds={} distance={}m'.format(p['approach_direction'], p['approach_speed'], p['approach_seconds'], p['approach_distance_m']))
    base_drive(p['approach_direction'], p['approach_speed'], p['approach_seconds'], 'approach_can_distance_{}m'.format(p['approach_distance_m']))


def turn_to_drop():
    p = current_motion_params()
    enter_state('turn_to_drop', 'direction={} speed={} seconds={} stop_angle={}deg'.format(p['drop_turn_direction'], p['drop_turn_speed'], p['drop_turn_seconds'], p['drop_stop_angle_deg']))
    base_drive(p['drop_turn_direction'], p['drop_turn_speed'], p['drop_turn_seconds'], 'drop_turn_to_angle_{}'.format(p['drop_stop_angle_deg']))


def drive_to_drop():
    p = current_motion_params()
    enter_state('drive_to_drop', 'direction={} speed={} seconds={} distance={}m'.format(p['drop_drive_direction'], p['drop_drive_speed'], p['drop_drive_seconds'], p['drop_drive_distance_m']))
    base_drive(p['drop_drive_direction'], p['drop_drive_speed'], p['drop_drive_seconds'], 'drive_to_drop_distance_{}m'.format(p['drop_drive_distance_m']))


def run_motion_pickup_demo(_=None):
    p = current_motion_params()
    print('[flow] motion pickup demo start')
    print('[flow] dry_run_base={} dry_run_arm={}'.format(p['dry_run_base'], p['dry_run_arm']))
    print('[target] no camera mode; scan and approach parameters stand in for can detection')
    print_points()
    try:
        safe_home()
#         state_pause('safe_home')
        time.sleep(3.0)
        base_stop()

        print('[flow] rotate in place to simulate target scan')
        scan_for_can()

        print('[flow] hardcoded move from A to B')
        approach_can()

        print('[flow] grab can')
        grab_sequence_without_final_home()

        print('[flow] lift_state_carry is carrying posture; move to drop point')
        turn_to_drop()

        print('[flow] hardcoded move from B to C')
        drive_to_drop()

        print('[flow] release at drop point')
        drop_release_sequence()
        print('[flow] demo done')
        return True
    finally:
        print('[flow] final cleanup: stop base and safe_home')
        base_stop()
        safe_home()


def emergency_stop(_=None):
    print('[stop] emergency stop')
    base_stop()
    safe_home()

In [16]:
save_button = widgets.Button(description='Save Params', button_style='info')
point_button = widgets.Button(description='Print Points', button_style='')
approach_button = widgets.Button(description='Approach Can', button_style='')
drive_drop_button = widgets.Button(description='Drive To Drop', button_style='')
run_button = widgets.Button(description='Run Full Demo', button_style='success')
stop_button = widgets.Button(description='Emergency Stop', button_style='danger')
home_button = widgets.Button(description='Safe Home', button_style='warning')
open_button = widgets.Button(description='Open Gripper', button_style='')
reaching_button = widgets.Button(description='Reaching State', button_style='')
lift_grip_button = widgets.Button(description='Lift Grip', button_style='')
lift_carry_button = widgets.Button(description='Lift Carry', button_style='')
drop_button = widgets.Button(description='Drop Release', button_style='')
clear_log_button = widgets.Button(description='Clear Log', button_style='')

control_buttons = [
    save_button, point_button, approach_button, drive_drop_button, run_button,
    home_button, open_button, reaching_button, lift_grip_button, lift_carry_button, drop_button,
]

log_output = widgets.Output(layout={
    'border': '1px solid #bbb',
    'height': '260px',
    'overflow_y': 'auto',
    'width': '100%',
})


def run_with_log(func):
    def wrapped(_=None):
        global operation_busy
        with log_output:
            if operation_busy:
                print('[busy] another operation is running; ignored')
                return
            operation_busy = True
            set_controls_disabled(True)
            try:
                func()
            except Exception as exc:
                print('[error]', type(exc).__name__ + ':', exc)
            finally:
                operation_busy = False
                set_controls_disabled(False)
    return wrapped


def clear_log(_=None):
    log_output.clear_output()


save_button.on_click(run_with_log(lambda: save_all_params()))
point_button.on_click(run_with_log(print_points))
approach_button.on_click(run_with_log(approach_can))
drive_drop_button.on_click(run_with_log(drive_to_drop))
run_button.on_click(run_with_log(lambda: run_motion_pickup_demo()))
stop_button.on_click(run_with_log(lambda: emergency_stop()))
home_button.on_click(run_with_log(safe_home))
open_button.on_click(run_with_log(open_gripper))
reaching_button.on_click(run_with_log(reaching_state))
lift_grip_button.on_click(run_with_log(lift_state_grip))
lift_carry_button.on_click(run_with_log(lift_state_carry))
drop_button.on_click(run_with_log(drop_release_sequence))
clear_log_button.on_click(clear_log)

button_row_1 = widgets.HBox([save_button, point_button, approach_button, drive_drop_button])
button_row_2 = widgets.HBox([home_button, open_button, reaching_button, lift_grip_button, lift_carry_button, drop_button])
button_row_3 = widgets.HBox([run_button, stop_button, clear_log_button])

ui = widgets.VBox([
    button_row_1,
    button_row_2,
    button_row_3,
    widgets.HTML('<b>Log output</b>'),
    log_output,
    widgets.HTML('<b>Safety</b>'),
    widgets.HBox([motion_widgets['dry_run_base'], motion_widgets['dry_run_arm']]),
    widgets.HTML('<b>Arm state isolation</b>'),
    grab_widgets['state_pause_seconds'],
    grab_widgets['reaching_pause_seconds'],
    widgets.HTML('<b>Hardcoded points on floor</b>'),
    widgets.HBox([motion_widgets['start_x'], motion_widgets['start_y']]),
    widgets.HBox([motion_widgets['can_x'], motion_widgets['can_y']]),
    widgets.HBox([motion_widgets['drop_x'], motion_widgets['drop_y']]),
    widgets.HTML('<b>Scan turn and approach to can</b>'),
    widgets.HBox([motion_widgets['scan_turn_direction'], motion_widgets['scan_stop_angle_deg']]),
    widgets.HBox([motion_widgets['scan_turn_speed'], motion_widgets['scan_turn_seconds']]),
    widgets.HBox([motion_widgets['approach_direction'], motion_widgets['approach_distance_m']]),
    widgets.HBox([motion_widgets['approach_speed'], motion_widgets['approach_seconds']]),
    widgets.HTML('<b>Turn and drive to drop point</b>'),
    widgets.HBox([motion_widgets['drop_turn_direction'], motion_widgets['drop_stop_angle_deg']]),
    widgets.HBox([motion_widgets['drop_turn_speed'], motion_widgets['drop_turn_seconds']]),
    widgets.HBox([motion_widgets['drop_drive_direction'], motion_widgets['drop_drive_distance_m']]),
    widgets.HBox([motion_widgets['drop_drive_speed'], motion_widgets['drop_drive_seconds']]),
])

display(ui)

## Usage

1. Run the code cells above until the UI appears.
2. Click Print Points to inspect the hardcoded A/B/C floor points.
3. Use Approach Can and Drive To Drop to test those hardcoded base movements independently.
4. First run should keep dry_run_base and dry_run_arm enabled, then click Run Full Demo and inspect the logs.
5. Full grab order is Open Gripper, Reaching State, Lift Grip, Lift Carry.
6. state_pause_seconds separates complete arm states; settle_seconds still separates individual servo commands.
7. After the parameters look right, disable the relevant dry-run checkbox and run again. Keep your hand ready for Emergency Stop during real movement.